In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
arkansas_files=["arkansas_batch1.csv","arkansas_batch2.csv","arkansas_other_fixed.csv"]
arkansas_sampling_plan={0: 616, 5: 4677, 3: 2423, 1: 1522, 2: 762}

cal_files = [
    "../california_alfafa.csv", "../california_almonds.csv", "../california_grapes.csv",
    "../california_pistachios.csv", "../california_rice.csv", "../california_other_fixed.csv"
]
california_sampling_plan={0:3512, 69:2054, 3:2037, 36:974, 75:783, 204:640}

## merge batches

### califorina

In [5]:

data_frames = [pd.read_csv(f) for f in cal_files]

# 2. Combine all rows into one giant dataframe immediately
# This puts all Alfalfa rows first, then Almonds, etc.
merged_all = pd.concat(data_frames, axis=0, ignore_index=True)

# 3. GLOBAL SORT
# Sorting by 'time' creates the 36 blocks.
# Sorting by '.geo' within each time block ensures the physical points 
# stay in the exact same order across all 36 periods.
merged_stacked = merged_all.sort_values(['time', '.geo']).reset_index(drop=True)

# 4. Verification
print(merged_stacked.info())
print("Shape of merged data:", merged_stacked.shape)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3967452 entries, 0 to 3967451
Data columns (total 16 columns):
 #   Column        Dtype  
---  ------        -----  
 0   system:index  object 
 1   B11           float64
 2   B12           float64
 3   B2            float64
 4   B3            float64
 5   B4            float64
 6   B5            float64
 7   B6            float64
 8   B7            float64
 9   B8            float64
 10  B8A           float64
 11  SCL           float64
 12  crop          int64  
 13  cropland      int64  
 14  time          object 
 15  .geo          object 
dtypes: float64(11), int64(2), object(3)
memory usage: 484.3+ MB
None
Shape of merged data: (3967452, 16)


In [6]:
batch_cal = merged_stacked.to_numpy()

print(merged_stacked.info())
print("Shape of merged data:", batch_cal.shape)

batch1_cal=merged_stacked.to_numpy()[:int(merged_stacked.shape[0]/36)]
np.unique(batch1_cal[:,12], return_counts=True)  


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3967452 entries, 0 to 3967451
Data columns (total 16 columns):
 #   Column        Dtype  
---  ------        -----  
 0   system:index  object 
 1   B11           float64
 2   B12           float64
 3   B2            float64
 4   B3            float64
 5   B4            float64
 6   B5            float64
 7   B6            float64
 8   B7            float64
 9   B8            float64
 10  B8A           float64
 11  SCL           float64
 12  crop          int64  
 13  cropland      int64  
 14  time          object 
 15  .geo          object 
dtypes: float64(11), int64(2), object(3)
memory usage: 484.3+ MB
None
Shape of merged data: (3967452, 16)


(array([0, 3, 36, 69, 75, 204], dtype=object),
 array([ 4902,  6422, 18217, 26306,  2478, 51882]))

### arkansas

In [36]:
data1 = pd.read_csv("../arkansas_batch1.csv")
data2 = pd.read_csv("../arkansas_batch2.csv")
data3 = pd.read_csv("../arkansas_other_fixed.csv")
data1_sorted = data1.sort_values(['time', 'system:index']).reset_index(drop=True)
data2_sorted = data2.sort_values(['time', 'system:index']).reset_index(drop=True)
data3_sorted = data3.sort_values(['time', 'system:index']).reset_index(drop=True)


# Get unique 10-day intervals
unique_times = data1_sorted['time'].unique()

# List to collect stacked blocks
stacked_blocks = []

for t in unique_times:
    block1 = data1_sorted[data1_sorted['time'] == t]
    block2 = data2_sorted[data2_sorted['time'] == t]
    block3 = data3_sorted[data3_sorted['time'] == t]
    
    # Keep the order of points as they appear in each block
    stacked_block = pd.concat([block1, block2, block3], axis=0, ignore_index=True)
    stacked_blocks.append(stacked_block)

# Combine all 10-day blocks
merged_stacked = pd.concat(stacked_blocks, axis=0, ignore_index=True)

# Convert to NumPy if needed
batch_ark = merged_stacked.to_numpy()

print(merged_stacked.info())
print("Shape of merged data:", batch_ark.shape)

batch1_ark=merged_stacked.to_numpy()[:int(merged_stacked.shape[0]/36)]
np.unique(batch1_ark[:,12], return_counts=True)  




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033596 entries, 0 to 1033595
Data columns (total 16 columns):
 #   Column        Non-Null Count    Dtype  
---  ------        --------------    -----  
 0   system:index  1033596 non-null  object 
 1   B11           1033596 non-null  float64
 2   B12           1033596 non-null  float64
 3   B2            1033596 non-null  float64
 4   B3            1033596 non-null  float64
 5   B4            1033596 non-null  float64
 6   B5            1033596 non-null  float64
 7   B6            1033596 non-null  float64
 8   B7            1033596 non-null  float64
 9   B8            1033596 non-null  float64
 10  B8A           1033596 non-null  float64
 11  SCL           1033596 non-null  float64
 12  crop          1033596 non-null  int64  
 13  cropland      1033596 non-null  int64  
 14  time          1033596 non-null  object 
 15  .geo          1033596 non-null  object 
dtypes: float64(11), int64(2), object(3)
memory usage: 126.2+ MB
None
Shape o

(array([0, 1, 2, 3, 5], dtype=object),
 array([ 2717,  1701,  1929,  3679, 18685]))

## remove extra (preference to those with many missing values)

### check for data alignment

In [37]:
# Assuming 36 time steps
n_steps = 36
n_total_rows = merged_stacked.shape[0]
n_points = int(n_total_rows / n_steps)

# Split the .geo column into 36 chunks (one for each time interval)
geo_column = merged_stacked['.geo'].values
chunks = [geo_column[i * n_points : (i + 1) * n_points] for i in range(n_steps)]

# Check if all chunks are identical to the first chunk
is_aligned = all(np.array_equal(chunks[0], chunk) for chunk in chunks)

if is_aligned:
    print(f"✅ Data is PERFECTLY aligned. Each block has {n_points} identical points in order.")
else:
    print("❌ Data is NOT aligned. Points are missing or ordered differently in some time steps.")

✅ Data is PERFECTLY aligned. Each block has 28711 identical points in order.


### reshape

In [38]:
import json
import pandas as pd

# 1. Parse the .geo column to extract coordinates
# The coordinates are in the format [longitude, latitude]
def extract_coords(geo_str):
    geo_dict = json.loads(geo_str)
    return pd.Series(geo_dict['coordinates'])

# Apply the extraction: index 0 is longitude, index 1 is latitude
merged_stacked[['longitude', 'latitude']] = merged_stacked['.geo'].apply(extract_coords)

# 2. Adjust the drop logic
# We keep longitude and latitude, but still drop the original .geo and other non-features
features_df = merged_stacked.drop(columns=['.geo', 'time', 'system:index', 'cropland']) 

# 3. Convert to NumPy
# Note: If you want exactly 10 bands + 2 coordinates, your features count is now 12
raw_array = features_df.to_numpy()

# 4. Reshape: (Time, Points, Features) -> (Points, Time, Features)
# Update the third dimension in reshape to match your actual column count (e.g., 12)
n_features = features_df.shape[1] 
final_data = raw_array.reshape(n_steps, n_points, n_features).transpose(1, 0, 2)

print("Final Shape:", final_data.shape) 
print("First point's features (including long/lat):\n", final_data[0, 0])

Final Shape: (28711, 36, 14)
First point's features (including long/lat):
 [1604.         1027.          406.          494.          814.
  993.         1140.         1305.         1592.         1569.
    5.            5.          -91.33256834   34.91791934]


In [47]:
final_data[:,:,10:]

array([[[ 5.00000000e+00,  5.00000000e+00, -9.13325683e+01,
          3.49179193e+01],
        [ 5.00000000e+00,  5.00000000e+00, -9.13325683e+01,
          3.49179193e+01],
        [ 8.00000000e+00,  5.00000000e+00, -9.13325683e+01,
          3.49179193e+01],
        ...,
        [ 5.00000000e+00,  5.00000000e+00, -9.13325683e+01,
          3.49179193e+01],
        [ 1.00000000e+01,  5.00000000e+00, -9.13325683e+01,
          3.49179193e+01],
        [ 5.00000000e+00,  5.00000000e+00, -9.13325683e+01,
          3.49179193e+01]],

       [[ 5.00000000e+00,  5.00000000e+00, -9.05109692e+01,
          3.49967914e+01],
        [ 5.00000000e+00,  5.00000000e+00, -9.05109692e+01,
          3.49967914e+01],
        [ 8.00000000e+00,  5.00000000e+00, -9.05109692e+01,
          3.49967914e+01],
        ...,
        [ 5.00000000e+00,  5.00000000e+00, -9.05109692e+01,
          3.49967914e+01],
        [ 8.00000000e+00,  5.00000000e+00, -9.05109692e+01,
          3.49967914e+01],
        [ 5.000

### interpolate missing values and pick final values

In [ ]:
"""
psudo code:

add a mask that indicates whether the data points has missing values
if target_num_samples>sample:
    sample=+clean data
    if target_num_samples>sample:
        order data with missing value from points with least to most missing values 
        complete samples with it until we reach target_num_samples
        interpolate the missing data
        
        
"""

In [49]:
from scipy.interpolate import interp1d
# ==========================================
# 1. PREPARATION & GAP SCORING
# ==========================================

# Calculate how many 10-day windows are missing (-9999) for each point
# final_data shape is (N, 36, 12)
gap_counts = np.sum(np.any(final_data[:, :, :10] == -9999, axis=2), axis=1)

# Prepare the Presence Mask (1.0 = Real Data, 0.0 = Gap)
# We use the first band (index 0) to check for gaps
presence_mask = (final_data[:, :, 0] != -9999).astype(np.float32).reshape(-1, 36, 1)

# ==========================================
# 2. INTERPOLATION FUNCTION
# ==========================================

def fill_time_series(data_3d):
    """
    Linearly interpolates missing values (-9999) across the time axis.
    """
    working_data = data_3d.copy().astype(np.float32)
    working_data[working_data == -9999] = np.nan
    
    n_points, n_steps, n_feats = working_data.shape
    
    for p in range(n_points):
        for f in range(10):
            y = working_data[p, :, f]
            nans = np.isnan(y)
            
            # If the entire year is missing for this band, fill with 0
            if np.all(nans):
                y[:] = 0 
            # If there are some gaps, interpolate them
            elif np.any(nans):
                x_known = np.where(~nans)[0]
                # Linear interpolation with extrapolation for start/end of year
                f_interp = interp1d(x_known, y[~nans], kind='linear', fill_value="extrapolate")
                y[nans] = f_interp(np.where(nans)[0])
            
            working_data[p, :, f] = y
    # 2. Handle Metadata/Coords (10 to n_feats) OUTSIDE the point loop
    # Since these are static values (Long/Lat/Label), we just replace NaNs with 0 
    # or the actual value if it exists.
    if n_feats > 10:
        meta_slice = working_data[:, :, 10:]
        working_data[:, :, 10:] = np.nan_to_num(meta_slice, nan=0.0)
    return working_data

# Run interpolation on everything to prepare the features
interpolated_all = fill_time_series(final_data)

# Add the presence mask as the 13th column
# Resulting shape: (N, 36, 13)
data_with_mask = np.concatenate([interpolated_all, presence_mask], axis=2)

# ==========================================
# 3. QUALITY-PRIORITIZED SAMPLING
# ==========================================

sampling_plan = arkansas_sampling_plan
crop_col_idx = 11 # The 'crop' label is at index 11
point_crop_ids = data_with_mask[:, 0, crop_col_idx]

final_samples = []

print("--- Starting Quality-Aware Sampling ---")
for crop_id, target_count in sampling_plan.items():
    # Find all available indices for this specific crop
    crop_indices = np.where(point_crop_ids == crop_id)[0]
    
    # SORT indices by gap count: points with 0 gaps come first, then 1, 2, etc.
    sorted_indices = crop_indices[np.argsort(gap_counts[crop_indices])]
    
    # Select the highest quality points (up to our target count)
    selected = sorted_indices[:target_count]
    
    # Calculate stats for the selection
    perfect_count = np.sum(gap_counts[selected] == 0)
    avg_gaps = np.mean(gap_counts[selected])
    
    print(f"Crop {crop_id}: Selected {len(selected)} points.")
    print(f"   -> {perfect_count} are perfect (0 gaps).")
    print(f"   -> Average gaps in selection: {avg_gaps:.2f}")
    
    final_samples.append(data_with_mask[selected])

# ==========================================
# 4. FINAL ASSEMBLY
# ==========================================

# Combine all classes into one dataset
balanced_dataset = np.concatenate(final_samples, axis=0)

# Shuffle the data so classes aren't grouped together
np.random.shuffle(balanced_dataset)

print("-" * 30)
print(f"Final Dataset Shape: {balanced_dataset.shape}")
print(f"Columns: Bands 1-10, SCL(11), Label(12), PresenceMask(13)")

--- Starting Quality-Aware Sampling ---
Crop 0: Selected 616 points.
   -> 0 are perfect (0 gaps).
   -> Average gaps in selection: 2.49
Crop 5: Selected 4677 points.
   -> 4513 are perfect (0 gaps).
   -> Average gaps in selection: 0.04
Crop 3: Selected 2423 points.
   -> 1046 are perfect (0 gaps).
   -> Average gaps in selection: 0.57
Crop 1: Selected 1522 points.
   -> 359 are perfect (0 gaps).
   -> Average gaps in selection: 0.96
Crop 2: Selected 762 points.
   -> 186 are perfect (0 gaps).
   -> Average gaps in selection: 0.76
------------------------------
Final Dataset Shape: (10000, 36, 15)
Columns: Bands 1-10, SCL(11), Label(12), PresenceMask(13)


### dataset quality stats

In [50]:
# 1. Extract the Presence Mask from the final dataset 
# (It's the very last column we concatenated)
final_presence_mask = balanced_dataset[:, :, -1] 

# 2. Count zeros (gaps) per point
# Since 1.0 = data and 0.0 = gap, (1 - mask) gives us 1s for gaps
gaps_per_point = np.sum(final_presence_mask == 0, axis=1)

# 3. Calculate statistics
avg_gaps = np.mean(gaps_per_point)
max_gaps = np.max(gaps_per_point)
min_gaps = np.min(gaps_per_point)

print(f"📊 Final Dataset Gap Stats:")
print(f"Average gaps: {avg_gaps:.2f} per point")
print(f"Max gaps in a single point: {max_gaps}")
print(f"Min gaps in a single point: {min_gaps}")

📊 Final Dataset Gap Stats:
Average gaps: 0.51 per point
Max gaps in a single point: 3
Min gaps in a single point: 0


## export as csv file

In [51]:
import pandas as pd
import numpy as np

# 1. Define the features that actually change over time
# We exclude 'crop', 'long', and 'lat' from the 36-step loop
# Indices in balanced_dataset: 0-9 (Bands), 10 (SCL), 14 (mask)
time_features = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12', 'mask']
time_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 14]

all_col_names = []
for t in range(36):
    for b in time_features:
        all_col_names.append(f"{b}_{t}")

# 2. Slice the dataset to get only the time-varying columns for flattening
# Shape change: (N, 36, 15) -> (N, 36, 12)
time_series_data = balanced_dataset[:, :, time_indices]

# 3. Reshape to 2D
# Shape change: (N, 36, 12) -> (N, 432)
flattened_data = time_series_data.reshape(time_series_data.shape[0], -1)

# 4. Create DataFrame
df_final = pd.DataFrame(flattened_data, columns=all_col_names)

# 5. Add Static Columns (One-time entry per point)
# These are identical across all 36 steps, so we take them from step 0
df_final['target_label'] = balanced_dataset[:, 0, 11]
df_final['longitude']    = balanced_dataset[:, 0, 12]
df_final['latitude']     = balanced_dataset[:, 0, 13]

# 6. Save
df_final.to_csv("arkansas_coord.csv", index=False)

print(f"✅ Successfully saved {len(df_final)} samples.")
print(f"Total time-series columns: {flattened_data.shape[1]}")
print(f"Total CSV columns (incl. metadata): {len(df_final.columns)}")

✅ Successfully saved 10000 samples.
Total time-series columns: 396
Total CSV columns (incl. metadata): 399


### import csv script

read data for part 2 feature extraction

In [ ]:
# 1. Load the data
data = pd.read_csv("../arkansas_coord.csv")

# 2. Extract targets
y = data["target_label"].values

# 3. Drop the 3 static columns and reshape
features_only = data.drop(columns=["target_label", "longitude", "latitude"]).values
X_reshaped = features_only.reshape(-1, 36, 11) 

# 4. Extract
X_spectral = X_reshaped[:, :, :10] #input 1
X_mask = X_reshaped[:, :, 10]      #input 2

print(f"Reshape successful!")
print(f"X_spectral shape: {X_spectral.shape}") # Should be (10000, 36, 10)
print(f"X_mask shape:     {X_mask.shape}")     # Should be (10000, 36)

Reshape successful!
X_spectral shape: (10000, 36, 10)
X_mask shape:     (10000, 36)


read data for training

In [ ]:
data = pd.read_csv("../arkansas.csv")

# extract target
y = data["target_label"]
print (np.unique(y,return_counts=True))

# Remove the last column (target_label) and reshape to (10 000 , 36, 13)
features_only = data.values[:, :-1] 
original_shape = features_only.reshape(-1, 36, 13)

# Extract Spectral Bands (First 10 indices: B2 through B12)
X = original_shape[:, :, :10]
print(X[:5])

# 3. Extract Mask (The 13th index, which is index 12)
input2 = original_shape[:, :, 12]
print(input2[:5])